In [1]:
building = 0
time = 24
price = 'Realistic'
noise = 0
full_mpc = False

# Setup

In [2]:
import pickle
import numpy as np

import src.data.dataprep as prep
import src.data.featurisation as features

import src.optimization.pv_battery as pvb

In [3]:
# Import the base data and resample it from 5 minutes to hourly
nl_data = prep.dutch_data('../data/Dutchdata_clean/building_' + str(building) + '.parquet', 'h', price=price)

# Data Augmentation

In [4]:
# Add hourly sinusoidal patterns as features
featurisation = features.Featurisation(nl_data)
nl_data = featurisation.cyclic_features(yearly=False)[0]

In [5]:
# If we have no noise, the load forecast is perfect and actuals = forecast
if noise == 0:
    nl_data['load_fcst'] = nl_data['load']

# Else we take one of the forecasts based on the level of noise indicated in the first cell
else:
    load_fcst_data = np.load('../data/load_fcsts/building_' + str(building) + '_' + str(noise) + '_noise_forecast.npy')
    nl_data['load_fcst'] = load_fcst_data

In [6]:
# Include a DNI forecast
nl_data['direct_rad'] = np.load("../data/DHI_fcsts/DHI_fcst.npy")

In [7]:
# Include a battery based on Weniger et al. “Sizing of Residential PV Battery Systems,”
battery_capacity = round(nl_data['load'].resample('YE').sum().max() * 1.1 / 1000,1)

# The C-rate comes from Olivieri and McConky “Optimization of residential battery energy storage system scheduling for cost and emissions reductions,”
max_charge = battery_capacity/2.7
max_discharge = max_charge

In [8]:
# Set up the PV-B system
pvb_system = pvb.PV_battery(nl_data, building, battery_capacity, max_charge, max_discharge, self_consumption=False)

# Parameters

In [9]:
# Hyperparameters
layers = 3
neurons = 200
train_test_split = 0.6

# Parameters
past_features = ['solar_energy']
future_features = ['direct_rad', 'hour_sin','hour_cos']
opt_features = ['load', 'load_fcst','offtake','injection']

opt_future_features = future_features + opt_features

# Create domain_min and domain_max for the models
domain_min = [min(nl_data['solar_energy'][:676*24+24]), # past: solar energy
              min(nl_data['direct_rad'][:676*24+24]),   # future: direct radiation
              -1,-1,                                    # future: hour sin + cos
              0,0,0,0,                                  # future: optimization parameters
              0]                                        # target: solar energy

domain_max = [max(nl_data['solar_energy'][:676*24+24]), # past: solar energy
              max(nl_data['direct_rad'][:676*24+24]),   # future: direct radiation
              1,1,                                      # future: hour sin + cos
              1,1,1,1,                                  # future: optimization parameters
              1]                                        # target: solar energy

# Execute the optimization

In [10]:
# Do a single optimization at midnight
if full_mpc is False:
    dictionary_perfect = pvb_system.execute_optimization(time,23,'Perfect',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)[0]
    dictionary_naive = pvb_system.execute_optimization(time,23,'Naive',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)[0]
    dictionary_lstm = pvb_system.execute_optimization(time,23,'LSTM',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)[0]
    dictionary_cvx = pvb_system.execute_optimization(time,23,'CVX',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)[0]
    dictionary_lstm_cvx = pvb_system.execute_optimization(time,23,'LSTM_CVX',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)[0]

    all_models = {'perfect': dictionary_perfect,
                  'naive': dictionary_naive,
                  'lstm': dictionary_lstm,
                  'cvx': dictionary_cvx,
                  'lstm_cvx': dictionary_lstm_cvx}

# Do an optimization every hour, and execute the first scheduling action every time
else:
    dictionary_list_perfect = pvb_system.execute_optimization(time,0,'Perfect',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)
    dictionary_list_naive = pvb_system.execute_optimization(time,0,'Naive',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)
    dictionary_list_lstm = pvb_system.execute_optimization(time,0,'LSTM',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)
    dictionary_list_cvx = pvb_system.execute_optimization(time,0,'CVX',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)
    dictionary_list_lstm_cvx = pvb_system.execute_optimization(time,0,'LSTM_CVX',neurons,layers,past_features,opt_future_features,domain_min,domain_max,train_test_split=train_test_split, noise=noise)

    all_models = {'perfect': dictionary_list_perfect,
                  'naive': dictionary_list_naive,
                  'lstm': dictionary_list_lstm,
                  'cvx': dictionary_list_cvx,
                  'lstm_cvx': dictionary_list_lstm_cvx}

Setting up optimization for 0:00
Setting up optimization for 0:00
Setting up optimization for 0:00


C:\Users\jdepoort\Documents\Projects\Value_oriented_forecasting_for_PVB\Project\src\optimization\pv_battery.py:232: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(

FileNotFoundError: [Errno 2] No such file or directory: '../models/LSTM/building0_24h.pth'

In [ ]:
# Save the results
if full_mpc is False:
    with open('../results/optimisation/single_opt_building_' + str(building) + '_' + str(noise) + '_noise.pkl', "wb") as f:
        pickle.dump(all_models, f)
else:
    with open('../results/optimisation/full_mpc_building_' + str(building) + '_' + str(noise) + '_noise.pkl', "wb") as f:
        pickle.dump(all_models, f)